In [2]:
from collections import defaultdict
from pathlib import Path
import pandas as pd

# Load every clean match in chronological order
matches = pd.read_csv(
    "data/processed/epl_matches_clean.csv",
    parse_dates=["Date"]
).sort_values("Date").reset_index(drop=True)

# Each team begins with the same strength rating
ratings = defaultdict(lambda: 1500.0)

K_FACTOR = 20
HOME_ADVANTAGE = 60

elo_rows = []

for _, match in matches.iterrows():
    home_team = match["HomeTeam"]
    away_team = match["AwayTeam"]

    # Ratings BEFORE this match begins
    home_elo = ratings[home_team]
    away_elo = ratings[away_team]

    elo_rows.append({
        "Season": match["Season"],
        "Date": match["Date"],
        "HomeTeam": home_team,
        "AwayTeam": away_team,
        "home_elo": home_elo,
        "away_elo": away_elo,
        "elo_difference": home_elo - away_elo,
    })

    # Expected chance of the home team winning, including home advantage
    expected_home = 1 / (
        1 + 10 ** ((away_elo - (home_elo + HOME_ADVANTAGE)) / 400)
    )

    # Actual match result from the home team's perspective
    if match["FTHG"] > match["FTAG"]:
        actual_home = 1.0
    elif match["FTHG"] < match["FTAG"]:
        actual_home = 0.0
    else:
        actual_home = 0.5

    # Update ratings only AFTER recording this match's features
    ratings[home_team] += K_FACTOR * (actual_home - expected_home)
    ratings[away_team] += K_FACTOR * ((1 - actual_home) - (1 - expected_home))

elo_data = pd.DataFrame(elo_rows)

# Add Elo values to the existing five-match form features
base_features = pd.read_csv(
    "data/processed/epl_features.csv",
    parse_dates=["Date"]
)

enhanced_features = base_features.merge(
    elo_data,
    on=["Season", "Date", "HomeTeam", "AwayTeam"],
    how="left",
    validate="one_to_one"
)

output_file = Path("data/processed/epl_enhanced_features.csv")
enhanced_features.to_csv(output_file, index=False)

print(f"Rows saved: {len(enhanced_features)}")
print("\nMissing Elo values:")
print(enhanced_features[["home_elo", "away_elo", "elo_difference"]].isna().sum())

display(
    enhanced_features[
        ["Date", "HomeTeam", "AwayTeam", "home_elo", "away_elo", "elo_difference"]
    ].head()
)

Rows saved: 4106

Missing Elo values:
home_elo          0
away_elo          0
elo_difference    0
dtype: int64


,Date,HomeTeam,AwayTeam,home_elo,away_elo,elo_difference
0,2015-09-19,Aston Villa,West Brom,1482.048516,1489.304258,-7.255743
1,2015-09-19,Bournemouth,Sunderland,1483.522522,1469.667036,13.855487
2,2015-09-19,Chelsea,Arsenal,1482.507109,1518.421081,-35.913972
3,2015-09-19,Man City,West Ham,1549.612261,1508.033525,41.578736
4,2015-09-19,Newcastle,Watford,1473.350808,1499.382680,-26.031872
